# 1113 51일차

## 1. softmax

여러 값을 한꺼번에 받아 합이 1인 비율로 바꾸는 함수 (`keras78_09_softmax`)

다중 분류 출력층은 클래스마다 점수를 내놓는데, 그 점수를 서로 견줄 수 있는 비율로 바꿔야 확률로 읽음

1. `softmax = lambda x: np.exp(x) / np.sum(np.exp(x))` : 각 값을 지수로 바꾸고 전체 합으로 나눔
2. `np.arange(1, 5)` : 값 네 개를 넣어 비율을 봄
3. `plt.pie(ratio, shadow=True, startangle=90)` : 비율이라 파이 차트로 그림

- 값 하나씩 바꾸는 다른 활성화 함수와 달리 묶음 전체를 보고 나눔. x-y 선 그래프로는 못 그림
- **합이 1이라 그대로 확률로 읽음**
- `np.exp`를 거쳐서 값이 조금만 커도 비율은 훨씬 커짐
- 출력층 `Dense(클래스 수, activation='softmax')`가 하는 계산이 이것
- `startangle`은 첫 조각이 시작하는 각도. 90이면 12시 방향에서 시작함

## 2. 경사하강법

기울기를 보고 그 반대 방향으로 조금씩 옮겨 가며 최솟값을 찾는 것 (`keras78_gradient1`, `keras78_gradient2`)

loss가 가장 작은 지점을 식으로 풀지 않고, 지금 위치의 기울기만 가지고 찾아감

1. `f = lambda x : x **2 - 4 * x + 6` : 최솟값이 하나인 2차 함수로 실험함
2. `gradient = lambda x : 2 * x - 4` : `f`를 미분한 식
3. `x = x - learning_rate * gradient(x)` : 기울기의 반대 방향으로 이동함
4. `x = -10.0`, `epochs = 50`, `learning_rate = 0.1` : 시작 위치, 반복 횟수, 이동 폭을 정하는 값
5. `"{:02d}\t {:6.5f}\t {:6.5f}\t".format(i + 1, x, f(x))` : 정수 두 자리와 소수점 다섯 자리로 자리를 맞춰 찍음

- 기울기가 양수면 왼쪽, 음수면 오른쪽으로 감. 빼기 때문에 방향이 저절로 정해짐
- **최솟값에 가까워질수록 기울기가 작아져서 이동 폭도 같이 줄어듦**
- 시작 위치가 멀어도 반복하면 같은 지점으로 모임
- `plt.plot(2, 2, 'sk', color='red')`로 최솟값 자리를 찍어 그래프와 맞춰 봄

## 3. learning_rate

한 번에 얼마나 옮길지 정하는 값 (`keras79_learning_rate`)

가중치를 직접 양쪽으로 옮겨 보면 `learning_rate`가 결과를 어떻게 바꾸는지 보임

1. `hypothesis = x * w`와 `loss = (hypothesis - y) ** 2` : 예측값과 mse
2. `up_predict = x * (w + lr)`, `down_predict = x * (w - lr)` : 양쪽으로 한 칸씩 가 본 예측
3. `if(up_loss > down_loss): w = w - lr`, `else: w = w + lr` : loss가 작은 쪽으로 옮김

- 미분 대신 양쪽 loss를 견줘서 방향만 정한 것
- `lr`이 크면 적은 반복으로 가까워지지만 정답을 지나쳐 왔다 갔다 함
- `lr`이 작으면 촘촘하게 가는 대신 반복이 많이 필요함
- **이동 폭이 늘 `lr`로 같아서 최솟값에 닿아도 멈추지 않고 그 간격으로 진동함**
- 경사하강법은 `lr * 기울기`만큼 옮기므로 가까워지면 이동 폭이 줄어듦

## 4. TensorBoard

학습 과정을 파일로 남겨 브라우저에서 그래프로 보는 도구 (`keras80_Tensorboard`)

콘솔에는 epoch마다 숫자만 지나가서 loss가 어디서 꺾였는지 보기 어려움

1. `from tensorflow.keras.callbacks import TensorBoard` : 콜백으로 들어 있음
2. `TensorBoard(log_dir="C:/study/_save/tensorboard/_keras80/", histogram_freq=0, write_graph=True, write_images=True)` : 로그를 남길 폴더와 무엇을 남길지 정함
3. `callbacks=[es, tb]` : `fit`에 다른 콜백과 같이 넘김
4. `tensorboard --logdir=.` : cmd에서 로그 폴더로 들어가 실행함
5. `http://localhost:6006` : 브라우저로 열어 봄

- `log_dir` 아래에 `train`·`validation` 폴더가 생기고 그 안에 로그 파일이 쌓임
- **`--logdir`에 적는 경로는 로그 폴더. 다른 폴더에서 띄우면 화면이 빈 채로 나옴**
- 폴더로 들어가지 않고 `tensorboard --logdir=C:/study/_save/tensorboard/_keras80`처럼 경로를 그대로 적어도 됨
- `write_graph=True`면 모델 구조가 GRAPHS 탭에 같이 남음
- `histogram_freq=0`이면 가중치 분포는 남기지 않음

#### cf) 로그 폴더

- 돌릴 때마다 `train`·`validation`에 파일이 하나씩 더 쌓임
- 한 폴더에 여러 번의 로그가 섞이면 한 그래프에 선이 겹쳐 그려짐
- 파일 이름 끝 숫자가 그 실행의 프로세스 번호라, 짝이 없는 파일을 찾아 지움
- **실행마다 `log_dir`을 다르게 주면 섞이지 않음**
- `_save/` 아래에 두면 저장소에 올라가지 않음